In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_silver = spark.table("internet_fijo_elt.silver.conexiones_internet_fijo")

display(df_silver.limit(10))

In [0]:
dim_ubicacion = (
    df_silver
    .select(
        "departamento",
        "provincia",
        "distrito"
    )
    .distinct()
    .withColumn(
        "ubicacion_id",
        F.row_number().over(
            Window.orderBy(
                "departamento",
                "provincia",
                "distrito"
            )
        )
    )
)

In [0]:
dim_ubicacion_renamed = dim_ubicacion.select(
    F.col("ubicacion_id").alias("id_ubicacion"),
    F.col("departamento"),
    F.col("provincia"),
    F.col("distrito")
)

dim_ubicacion_renamed.write.format("delta").mode("overwrite").saveAsTable(
    "internet_fijo_elt.gold.dim_ubicacion"
)

In [0]:
print("Dim Ubicación:", dim_ubicacion.count())

In [0]:
display(dim_ubicacion)